In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import precision_recall_curve, precision_score, recall_score, f1_score
from scipy.stats import mannwhitneyu
import shap

In [3]:
df = pd.read_csv("churnData_Fixed.csv")
 
leak_cols = ["signup_date", "year_month", "window_id"]
model_features = [c for c in df.columns if c not in leak_cols + ["Churn"]]
 
N_WINDOWS = df["window_id"].nunique()
print(f"Loaded {df.shape[0]} rows across {N_WINDOWS} windows, {len(model_features)} features")

Loaded 7043 rows across 10 windows, 32 features


In [4]:
window0 = df[df["window_id"] == 0]
X_window0 = window0[model_features]
y_window0 = window0["Churn"]
 
neg_count = (y_window0 == 0).sum()
pos_count = (y_window0 == 1).sum()
scale_pos_weight = neg_count / pos_count
 
u_model = CalibratedClassifierCV(
    XGBClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        eval_metric="logloss", random_state=42,
        scale_pos_weight=scale_pos_weight,
    ),
    method="sigmoid",
    cv=5,
)
u_model.fit(X_window0, y_window0)
 
print(f"\nU(x) model trained on window 0 ({len(X_window0)} customers, "
      f"scale_pos_weight={scale_pos_weight:.2f})")
 


U(x) model trained on window 0 (705 customers, scale_pos_weight=23.31)


In [5]:
window1 = df[df["window_id"] == 1]
X_window1 = window1[model_features]
y_window1 = window1["Churn"]
 
u_window1 = u_model.predict_proba(X_window1)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_window1, u_window1)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
DECISION_THRESHOLD = thresholds[f1_scores[:-1].argmax()]
print(f"Decision threshold (F1-optimal on window 0->1): {DECISION_THRESHOLD:.3f}")
 

Decision threshold (F1-optimal on window 0->1): 0.130


In [6]:
shap_base_model = XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.05,
    eval_metric="logloss", random_state=42,
    scale_pos_weight=scale_pos_weight,
)
shap_base_model.fit(X_window0, y_window0)
 
explainer = shap.TreeExplainer(shap_base_model)
 
shap_values_by_window = {}
for w in sorted(df["window_id"].unique()):
    window_data = df[df["window_id"] == w][model_features]
    sv = explainer.shap_values(window_data)
    shap_values_by_window[w] = pd.DataFrame(sv, columns=model_features)
 
baseline_shap = shap_values_by_window[0]
baseline_mean = baseline_shap.mean()
baseline_std = baseline_shap.std().replace(0, 1e-10)  # avoid divide-by-zero on constant features
 
print(f"\nSHAP values recomputed for all {N_WINDOWS} windows")
 
 
def rank_biserial_effect_size(n1, n2, u_stat):
    return 1 - (2 * u_stat) / (n1 * n2)
 
 
ALPHA = 0.05
effect_size_by_window = {}  # window -> {feature: effect_size}
for w in sorted(df["window_id"].unique()):
    if w == 0:
        continue
    window_shap = shap_values_by_window[w]
    sizes = {}
    for feature in model_features:
        stat, p_value = mannwhitneyu(baseline_shap[feature], window_shap[feature], alternative="two-sided")
        sizes[feature] = abs(rank_biserial_effect_size(len(baseline_shap), len(window_shap), stat))
    effect_size_by_window[w] = sizes
 
print("Per-window, per-feature effect sizes recomputed (Phase 3 logic)")
 
 


SHAP values recomputed for all 10 windows
Per-window, per-feature effect sizes recomputed (Phase 3 logic)


In [7]:
def compute_instance_drift_severity(window_shap_df, feature_weights):
    """
    window_shap_df   : SHAP values for the customers being scored (one row
                        per customer, one column per feature)
    feature_weights   : dict of {feature: population-level effect size for
                        this window}, from Step 2
    Returns a raw (unnormalized) D(x) score per customer.
    """
    z_scores = (window_shap_df - baseline_mean) / baseline_std
    weights = pd.Series(feature_weights)[model_features]
    weighted_abs_z = z_scores.abs().mul(weights, axis=1)
    return weighted_abs_z.sum(axis=1)
 
 
def normalize_0_1(series):
    """Min-max scale a per-window score to [0, 1] so it's comparable to U(x)."""
    span = series.max() - series.min()
    if span == 0:
        return series * 0
    return (series - series.min()) / span

In [8]:
DEFERRAL_BUDGET = 0.15  # top 15% of cases get deferred to a human analyst
LAMBDA_GRID = np.arange(0.0, 1.05, 0.05)  # bounded -- see note above
 
def get_window_u_d_error(w):
    window_data = df[df["window_id"] == w]
    X_w = window_data[model_features]
    y_w = window_data["Churn"].values
    u_w = u_model.predict_proba(X_w)[:, 1]
    d_w_raw = compute_instance_drift_severity(shap_values_by_window[w], effect_size_by_window[w])
    d_w = normalize_0_1(d_w_raw).values
    y_pred_w = (u_w >= DECISION_THRESHOLD).astype(int)
    is_error_w = (y_pred_w != y_w).astype(int)
    return u_w, d_w, is_error_w
 
u_val_1, d_val_1, err_val_1 = get_window_u_d_error(1)
u_val_2, d_val_2, err_val_2 = get_window_u_d_error(2)
 
print(f"\nValidation windows: window 1 has {err_val_1.sum()} model errors / {len(err_val_1)} customers, "
      f"window 2 has {err_val_2.sum()} / {len(err_val_2)}")
 
def deferred_flags_for_budget(score, budget):
    n = max(1, int(len(score) * budget))
    top_k_idx = np.argsort(-score)[:n]
    flags = np.zeros(len(score), dtype=int)
    flags[top_k_idx] = 1
    return flags
 
lambda_search_results = []
for lam in LAMBDA_GRID:
    score_1 = u_val_1 + d_val_1 + lam * u_val_1 * d_val_1
    score_2 = u_val_2 + d_val_2 + lam * u_val_2 * d_val_2
    deferred_1 = deferred_flags_for_budget(score_1, DEFERRAL_BUDGET)
    deferred_2 = deferred_flags_for_budget(score_2, DEFERRAL_BUDGET)
 
    # Pool both validation windows together for one combined metric, so
    # lambda is chosen for consistency across windows, not just one
    combined_errors = np.concatenate([err_val_1, err_val_2])
    combined_deferred = np.concatenate([deferred_1, deferred_2])
 
    prec = precision_score(combined_errors, combined_deferred, zero_division=0)
    rec = recall_score(combined_errors, combined_deferred, zero_division=0)
    f1 = f1_score(combined_errors, combined_deferred, zero_division=0)
    lambda_search_results.append({"lambda": lam, "precision": prec, "recall": rec, "f1": f1})
 
lambda_search_df = pd.DataFrame(lambda_search_results)
best_row = lambda_search_df.loc[lambda_search_df["f1"].idxmax()]
LAMBDA = best_row["lambda"]
 
print(f"\n--- Lambda search results (validation = windows 1+2 combined, budget = top {int(DEFERRAL_BUDGET*100)}%) ---")
print(lambda_search_df)
print(f"\nSelected lambda = {LAMBDA:.2f} "
      f"(deferral precision={best_row['precision']:.4f}, recall={best_row['recall']:.4f}, F1={best_row['f1']:.4f})")
 
 


Validation windows: window 1 has 147 model errors / 704 customers, window 2 has 187 / 704

--- Lambda search results (validation = windows 1+2 combined, budget = top 15%) ---
    lambda  precision    recall        f1
0     0.00   0.476190  0.299401  0.367647
1     0.05   0.476190  0.299401  0.367647
2     0.10   0.476190  0.299401  0.367647
3     0.15   0.476190  0.299401  0.367647
4     0.20   0.485714  0.305389  0.375000
5     0.25   0.485714  0.305389  0.375000
6     0.30   0.485714  0.305389  0.375000
7     0.35   0.495238  0.311377  0.382353
8     0.40   0.495238  0.311377  0.382353
9     0.45   0.500000  0.314371  0.386029
10    0.50   0.504762  0.317365  0.389706
11    0.55   0.514286  0.323353  0.397059
12    0.60   0.514286  0.323353  0.397059
13    0.65   0.514286  0.323353  0.397059
14    0.70   0.514286  0.323353  0.397059
15    0.75   0.519048  0.326347  0.400735
16    0.80   0.514286  0.323353  0.397059
17    0.85   0.514286  0.323353  0.397059
18    0.90   0.514286  0.3

In [9]:
routing_results = []
for w in sorted(df["window_id"].unique()):
    if w == 0:
        continue
    window_data = df[df["window_id"] == w]
    X_w = window_data[model_features]
    y_w = window_data["Churn"].values
 
    u_w = u_model.predict_proba(X_w)[:, 1]
    d_w_raw = compute_instance_drift_severity(shap_values_by_window[w], effect_size_by_window[w])
    d_w = normalize_0_1(d_w_raw)
 
    score_w = u_w + d_w + LAMBDA * u_w * d_w
 
    y_pred_w = (u_w >= DECISION_THRESHOLD).astype(int)
    is_error_w = (y_pred_w != y_w).astype(int)
 
    budget_n_w = max(1, int(len(score_w) * DEFERRAL_BUDGET))
    top_k_idx_w = np.argsort(-score_w)[:budget_n_w]
    deferred_flags_w = np.zeros(len(score_w), dtype=int)
    deferred_flags_w[top_k_idx_w] = 1
 
    routing_results.append({
        "window_id": w,
        "used_for_lambda_fit": w in (1, 2),
        "n_customers": len(y_w),
        "n_errors": is_error_w.sum(),
        "n_deferred": deferred_flags_w.sum(),
        "deferral_precision": precision_score(is_error_w, deferred_flags_w, zero_division=0),
        "deferral_recall": recall_score(is_error_w, deferred_flags_w, zero_division=0),
        "deferral_f1": f1_score(is_error_w, deferred_flags_w, zero_division=0),
    })
 
routing_df = pd.DataFrame(routing_results)
print(f"\n--- Deferral routing performance per window (budget = top {int(DEFERRAL_BUDGET*100)}%, lambda={LAMBDA:.2f}) ---")
print(routing_df.to_string())
 
routing_df.to_csv("phase4_routing_results.csv", index=False)
lambda_search_df.to_csv("phase4_lambda_search.csv", index=False)
print("\nSaved phase4_routing_results.csv and phase4_lambda_search.csv")
 
print("\nPhase 4 complete.")
print("`u_model`               -> Phase 2's calibrated classifier, U(x) source")
print("`shap_base_model`       -> frozen Phase 3 model, D(x) source")
print("`LAMBDA`                -> fitted interaction weight, chosen on windows 1+2 combined, bounded to [0,1]")
print("`DECISION_THRESHOLD`    -> used only to define 'model error' for fitting lambda, never applied to U(x) itself")
print("`routing_df`            -> per-window deferral precision/recall/F1, windows 3-9 are the true held-out results (also saved to CSV)")
 


--- Deferral routing performance per window (budget = top 15%, lambda=0.75) ---
   window_id  used_for_lambda_fit  n_customers  n_errors  n_deferred  deferral_precision  deferral_recall  deferral_f1
0          1                 True          704       147         105            0.447619         0.319728     0.373016
1          2                 True          704       187         105            0.590476         0.331551     0.424658
2          3                False          704       187         105            0.495238         0.278075     0.356164
3          4                False          705       217         105            0.504762         0.244240     0.329193
4          5                False          704       185         105            0.447619         0.254054     0.324138
5          6                False          704       205         105            0.447619         0.229268     0.303226
6          7                False          704       214         105            0.2857